In [ ]:
# 04_asymptotic_normality.ipynb
# Asymptotic normality of the contact count on the exhaustive counting board
# (midpoints seeded in C = [0, 1]^3, all pairwise contacts counted), 2r = 0.02.
# Produces the summary table, the skewness decay on log-log axes, and the
# histogram / Q-Q plot at the largest n. Raise `reps` (and n) for
# publication-quality estimates; the shipped figures use 8000 replicates for
# n <= 400 and 3500 for n = 800.
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update({"font.size": 10, "axes.edgecolor": "0.3"})
GRAY = "0.35"
FIG_DIR, TAB_DIR = "../results/figures", "../results/tables"
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(TAB_DIR, exist_ok=True)


def savefig(fig, name):
    fig.savefig(os.path.join(FIG_DIR, name + ".png"), dpi=600, bbox_inches="tight")
    fig.savefig(os.path.join(FIG_DIR, name + ".pdf"), bbox_inches="tight")


rng = np.random.default_rng(20261006)

ell = np.array([0.2, 0.4])
p = np.array([0.5, 0.5])
twor = 0.02


def seg_seg_dist(P1, P2, Q1, Q2):
    d1, d2, r = P2 - P1, Q2 - Q1, P1 - Q1
    a = (d1 * d1).sum(1); e = (d2 * d2).sum(1); f = (d2 * r).sum(1)
    b = (d1 * d2).sum(1); c = (d1 * r).sum(1)
    den = a * e - b * b
    s = np.where(den > 1e-12, np.clip((b * f - c * e) / den, 0, 1), 0.0)
    t = np.clip((b * s + f) / e, 0, 1)
    s = np.clip((b * t - c) / a, 0, 1)
    return np.linalg.norm((P1 + s[:, None] * d1) - (Q1 + t[:, None] * d2), axis=1)


def board_count(n, twor, rng):
    """Exhaustive counting board: midpoints ~ U(C), count all pairs with d <= 2r."""
    L = rng.choice(ell, size=n, p=p)
    X = rng.uniform(0.0, 1.0, size=(n, 3))
    v = rng.normal(size=(n, 3)); v /= np.linalg.norm(v, axis=1, keepdims=True)
    P1, P2 = X - 0.5 * L[:, None] * v, X + 0.5 * L[:, None] * v
    i, j = np.triu_indices(n, 1)
    return int(np.sum(seg_seg_dist(P1[i], P2[i], P1[j], P2[j]) <= twor))


# --- Summary over increasing n (raise reps toward 1e4 for publication) ---
reps = 3000
ns = [100, 200, 400, 800]
samples, rows = {}, []
for n in ns:
    c = np.array([board_count(n, twor, rng) for _ in range(reps)])
    samples[n] = c.astype(float)
    sk = stats.skew(c)
    rows.append({"n": n, "mean": c.mean(), "sd": c.std(ddof=1),
                 "skewness": sk, "excess_kurtosis": stats.kurtosis(c),
                 "skew_x_sqrt_n": sk * np.sqrt(n)})
table = pd.DataFrame(rows)
print(table.to_string(index=False))
table.to_latex(os.path.join(TAB_DIR, "table_normality.tex"), index=False, float_format="%.3f")

# --- Figure: skewness vs n on log-log axes with an n^{-1/2} reference line ---
nn = np.array(ns, dtype=float)
sk = np.abs(table["skewness"].values)
C = np.exp(np.mean(np.log(sk) + 0.5 * np.log(nn)))     # best-fit constant, slope fixed at -1/2
fig, ax = plt.subplots(figsize=(4.6, 3.2))
ax.loglog(nn, sk, marker="o", ls="none", color=GRAY, markeredgecolor="black",
          ms=7, label="simulated skewness")
xr = np.linspace(nn.min() * 0.9, nn.max() * 1.1, 50)
ax.loglog(xr, C / np.sqrt(xr), color="black", lw=1.3, label=r"$\propto n^{-1/2}$")
ax.set_xlabel(r"$n$")
ax.set_ylabel("skewness")
ax.legend(frameon=False)
savefig(fig, "fig_skewness")
plt.show()

# --- Figures: standardized histogram and normal Q-Q at the largest n ---
big = samples[ns[-1]]
z = (big - big.mean()) / big.std(ddof=1)
fig, ax = plt.subplots(figsize=(4.2, 3.2))
sns.histplot(z, bins=42, stat="density", color=GRAY, edgecolor="none", ax=ax)
xx = np.linspace(z.min(), z.max(), 200)
ax.plot(xx, stats.norm.pdf(xx), color="black", lw=1.4)
ax.set_xlabel(r"standardized $N_n$")
ax.set_ylabel("density")
savefig(fig, "fig_normal_hist")
plt.show()

fig, ax = plt.subplots(figsize=(4.2, 3.2))
osm, osr = stats.probplot(z, dist="norm")[0]
ax.scatter(osm, osr, s=9, facecolor=GRAY, edgecolor="none")
lim = [min(osm.min(), osr.min()), max(osm.max(), osr.max())]
ax.plot(lim, lim, color="black", lw=1.2)
ax.set_xlabel("theoretical quantiles")
ax.set_ylabel("sample quantiles")
savefig(fig, "fig_normal_qq")
plt.show()
